In [0]:
import os
from pathlib import Path
from typing import Literal

import cdsapi
import numpy as np
import pandas as pd
import xarray as xr
from tqdm.auto import tqdm
import ocha_stratus as stratus

prod_container_client = stratus.get_container_client(
    container_name="aa-data", stage="prod"
)
dev_container_client = stratus.get_container_client(
    container_name="projects", stage="dev"
)

In [0]:
GF_STATIONS = {
    # "wuroboki": {"lon": 12.767, "lat": 9.383},
    # "ibi":      {"lon": 9.725,  "lat": 8.175},
    "makurdi":  {"lon": 8.525,  "lat": 7.775},
    "umaisha":  {"lon": 7.175,  "lat": 7.975},
    "lokoja":   {"lon": 6.775,  "lat": 7.775},
    "baro":     {"lon": 6.425,  "lat": 8.575},
    "wuya":     {"lon": 5.825,  "lat": 9.075},
    "jebba":    {"lon": 4.875,  "lat": 9.175},
    "onitsha":  {"lon": 6.725,  "lat": 6.225},
    "yidere bode": {"lon": 4.125,  "lat": 11.375},
    "kende":   {"lon": 4.225,  "lat": 11.475},
}

In [0]:
def get_coords(station_name):
    station = GF_STATIONS[station_name]
    glofas_lon, glofas_lat = get_glofas_grid_coords(
        station["lon"], station["lat"]
    )
    pitch = 0.001
    N = glofas_lat + pitch
    S = glofas_lat
    E = glofas_lon + pitch
    W = glofas_lon
    return [N, W, S, E]

def download_glofas_reanalysis_to_blob(station_name: str):
    for year in tqdm(range(1979, 2025)):
        download_glofas_reanalysis_year_to_blob(year, station_name)

def get_blob_name(
    data_type: Literal["raw", "processed"],
    dataset: Literal["reanalysis", "reforecast", "forecast"],
    station_name: str,
    year: int = None,
) -> str:
    if year is None and data_type == "raw":
        raise ValueError("Year must be provided for raw data")
    if data_type == "raw":
        return f"ds-aa-nga-flooding/{data_type}/glofas/{dataset}/glofas_{data_type}_{dataset}_{station_name}_{year}.grib"  # noqa
    return f"ds-aa-nga-flooding/{data_type}/glofas/glofas_{dataset}_{station_name}.parquet"  # noqa

def get_glofas_grid_coords(lon, lat):
    grid_lat = np.arange(-90.025, 90, 0.05)
    grid_lon = np.arange(-180.025, 180, 0.05)
    nearest_lat_idx = (np.abs(grid_lat - lat)).argmin()
    nearest_lon_idx = (np.abs(grid_lon - lon)).argmin()
    return round(grid_lon[nearest_lon_idx], 3), round(
        grid_lat[nearest_lat_idx], 3
    )


def download_glofas_reanalysis_year_to_blob(
    year: int, station_name: str, pitch: float = 0.001, clobber: bool = False
):
    station = GF_STATIONS[station_name]
    glofas_lon, glofas_lat = get_glofas_grid_coords(
        station["lon"], station["lat"]
    )
    N = glofas_lat + pitch
    S = glofas_lat
    E = glofas_lon + pitch
    W = glofas_lon
    dataset = "cems-glofas-historical"
    request = {
        "system_version": ["version_4_0"],
        "hydrological_model": ["lisflood"],
        "product_type": ["consolidated"],
        "variable": ["river_discharge_in_the_last_24_hours"],
        "hyear": [f"{year}"],
        "hmonth": [f"{x:02}" for x in range(1, 13)],
        "hday": [f"{x:02}" for x in range(1, 32)],
        "data_format": "grib2",
        "download_format": "unarchived",
        "area": [N, W, S, E],
    }
    blob_name = get_blob_name("raw", "reanalysis", station_name, year)
    # check if blob exists
    if not clobber and check_blob_exists(blob_name):
        print(f"{blob_name} already exists in blob storage")
        return
    return download_raw_cds_api_to_blob(dataset, request, blob_name)


def download_raw_cds_api_to_blob(
    dataset: str,
    request: dict,
    blob_name: str,
    prod_dev: Literal["prod", "dev"] = "dev",
    keep_local_copy: bool = True,
):
    local_filepath = "temp" / Path(blob_name)
    if not local_filepath.parent.exists():
        os.makedirs(local_filepath.parent)

    # DON'T COMMIT
    c = cdsapi.Client(
        url="https://ewds.climate.copernicus.eu/api",
        key="3a27bef4-bf1a-4b76-982a-ca728901e2b4",
    )
    response = c.retrieve(dataset, request)
    response.download(local_filepath)
    with open(local_filepath, "rb") as file:
        stratus.upload_blob_data(file, blob_name, stage=prod_dev)
    if not keep_local_copy:
        os.remove(local_filepath)
    return local_filepath if keep_local_copy else None

def check_blob_exists(blob_name, prod_dev: Literal["prod", "dev"] = "dev"):
    if prod_dev == "dev":
        container_client = dev_container_client
    else:
        container_client = prod_container_client
    blob_client = container_client.get_blob_client(blob_name)
    return blob_client.exists()

In [0]:
from concurrent.futures import ThreadPoolExecutor, as_completed

max_workers = 4  # tune based on CDS API rate limits

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = {
        executor.submit(download_glofas_reanalysis_to_blob, station_name): station_name
        for station_name in GF_STATIONS
    }
    for future in as_completed(futures):
        station_name = futures[future]
        try:
            future.result()
            print(f"DONE: {station_name}")
        except Exception as e:
            print(f"FAILED: {station_name} — {e}")

makurdi
umaisha
lokoja
baro
wuya
jebba
onitsha
yidere bode
kende
